# 10 - Stage 2 Baseline and Ablation

This notebook trains conversation-level logistic regression baselines with fixed PAN-12 splits. Feature ablations are text only, stage aggregates, behavioral features, and all features combined.

Conversation IDs remain the evaluation unit. The supplied test split is kept untouched for final evaluation.

In [4]:
from pathlib import Path
import json
import joblib

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

WORKSPACE = Path.cwd()
PROCESSED_DIR = WORKSPACE / "data" / "processed"
REPORT_DIR = WORKSPACE / "reports"
MODEL_DIR = WORKSPACE / "models"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

FEATURES_PATH = PROCESSED_DIR / "pan12_conversation_features.csv"
if not FEATURES_PATH.exists():
    raise FileNotFoundError("Run 09_pan12_stage2_features.ipynb first.")

conversations = pd.read_csv(FEATURES_PATH)
conversations["conversation_text"] = conversations["conversation_text"].fillna("")
assert conversations["conversation_id"].is_unique
assert conversations["label"].isin([0, 1]).all()

stage_columns = [
    "fraction_predicted_200", "fraction_predicted_600", "fraction_predicted_900",
    "stage_600_threshold_onset", "stage_900_threshold_onset",
    "mean_stage_600_probability", "mean_stage_900_probability",
    "max_stage_600_probability", "max_stage_900_probability",
]
stage_columns = [column for column in stage_columns if column in conversations.columns]
behavioral_columns = [
    column for column in conversations.columns
    if column not in {"split", "conversation_id", "conversation_text", "label", *stage_columns}
]

print("Rows:", conversations.shape)
print(conversations.groupby("split")["label"].value_counts().to_string())
print("Stage features:", len(stage_columns))
print("Behavioral features:", len(behavioral_columns))

Rows: (222055, 21)
split  label
test   0        151391
       1          3737
train  0         64911
       1          2016
Stage features: 9
Behavioral features: 8


## Model variants

Numeric models use median imputation and standardization. Text uses word TF-IDF. Each ablation uses an internal validation threshold selected by maximum F1, then freezes that threshold for the supplied test split.

In [5]:
def build_numeric_pipeline(columns):
    numeric = ColumnTransformer([
        ("numeric", Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
        ]), columns),
    ])
    return Pipeline([
        ("features", numeric),
        ("classifier", LogisticRegression(class_weight="balanced", solver="lbfgs", max_iter=2000, random_state=RANDOM_SEED)),
    ])


def build_text_pipeline():
    return Pipeline([
        ("tfidf", TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=5, sublinear_tf=True, max_features=50000)),
        ("classifier", LogisticRegression(class_weight="balanced", solver="lbfgs", max_iter=2000, random_state=RANDOM_SEED)),
    ])


def build_combined_pipeline(columns):
    features = ColumnTransformer([
        ("text", TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=5, sublinear_tf=True, max_features=50000), "conversation_text"),
        ("numeric", Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler(with_mean=False)),
        ]), columns),
    ], sparse_threshold=0.3)
    return Pipeline([
        ("features", features),
        ("classifier", LogisticRegression(class_weight="balanced", solver="lbfgs", max_iter=2000, random_state=RANDOM_SEED)),
    ])

## Holdout threshold selection and test evaluation

The validation threshold is selected by maximum F1. It is then applied unchanged to the supplied PAN-12 test split.

In [6]:
train_frame = conversations.loc[conversations["split"] == "train"].copy()
test_frame = conversations.loc[conversations["split"] == "test"].copy()

train_inner, validation_inner = train_test_split(
    train_frame, test_size=0.20, random_state=RANDOM_SEED, stratify=train_frame["label"]
)


def choose_threshold(model, validation_frame, feature_values):
    validation_scores = model.predict_proba(feature_values(validation_frame))[:, 1]
    precision, recall, thresholds = precision_recall_curve(validation_frame["label"], validation_scores)
    f1_values = np.divide(
        2 * precision[:-1] * recall[:-1],
        precision[:-1] + recall[:-1],
        out=np.zeros_like(precision[:-1]),
        where=(precision[:-1] + recall[:-1]) != 0,
    )
    return float(thresholds[int(np.argmax(f1_values))])


results = []
models = {}
all_numeric_columns = stage_columns + behavioral_columns
variants = {
    "text_only": (build_text_pipeline(), "text"),
    "stage_only": (build_numeric_pipeline(stage_columns), "stage"),
    "behavior_only": (build_numeric_pipeline(behavioral_columns), "behavior"),
    "all": (build_combined_pipeline(all_numeric_columns), "all"),
}


def variant_features(frame, variant_name):
    if variant_name == "text":
        return frame["conversation_text"]
    if variant_name == "stage":
        return frame[stage_columns]
    if variant_name == "behavior":
        return frame[behavioral_columns]
    return frame


for variant_name, (model, feature_name) in variants.items():
    model.fit(variant_features(train_inner, feature_name), train_inner["label"])
    threshold = choose_threshold(
        model, validation_inner,
        lambda frame, feature_name=feature_name: variant_features(frame, feature_name),
    )
    test_scores = model.predict_proba(variant_features(test_frame, feature_name))[:, 1]
    test_predictions = (test_scores >= threshold).astype(int)
    y_test = test_frame["label"].to_numpy()
    results.append({
        "variant": variant_name,
        "threshold": threshold,
        "pr_auc": average_precision_score(y_test, test_scores),
        "roc_auc": roc_auc_score(y_test, test_scores),
        "precision": precision_score(y_test, test_predictions, zero_division=0),
        "recall": recall_score(y_test, test_predictions, zero_division=0),
        "f1": f1_score(y_test, test_predictions, zero_division=0),
    })
    models[variant_name] = model

ablation_results = pd.DataFrame(results)
display(ablation_results)
ablation_results.to_csv(REPORT_DIR / "stage2_ablation_results.csv", index=False)

joblib.dump(models["all"], MODEL_DIR / "stage2_logistic_all.joblib")
with (REPORT_DIR / "stage2_ablation_thresholds.json").open("w", encoding="utf-8") as handle:
    json.dump({row["variant"]: row["threshold"] for row in results}, handle, indent=2)

,variant,threshold,pr_auc,roc_auc,precision,recall,f1
0,text_only,0.812739,0.742247,0.984440,0.702305,0.676746,0.689289
1,stage_only,0.761646,0.105359,0.686303,0.180785,0.242708,0.207220
2,behavior_only,0.874426,0.210908,0.893933,0.259516,0.290072,0.273945
3,all,0.894720,0.747965,0.990301,0.705251,0.650522,0.676782


## Important limitations

* Text-only features may learn PAN-12 source or victim/prey vocabulary artifacts.
* Numeric features have PAN-12-only missingness patterns and should not be interpreted as real-world prevalence estimates.
* Conversation-level splits preserve independent conversations, but shared authors or sources can remain in both split sides.
* PR-AUC, thresholded precision, recall, and F1 should be reported together rather than accuracy alone.